#Data Processing

In [1]:
!apt-get install openjdk-8-jdk-headless -qq
!pip install pyspark

E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-8/openjdk-8-jre-headless_8u472-ga-1%7e22.04_amd64.deb  404  Not Found [IP: 91.189.92.22 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-8/openjdk-8-jdk-headless_8u472-ga-1%7e22.04_amd64.deb  404  Not Found [IP: 91.189.92.22 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviewsCleaning") \
    .config("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED") \
    .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .config("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS") \
    .getOrCreate()

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import numpy as np
import pandas as pd
import re

# Import Spark functions
from pyspark.sql.functions import *

##Review Data Processing

In [ ]:
# Review dataset columns
df_sample = pd.read_json(
    "/content/drive/MyDrive/Kpdl1/reviews_Electronics.jsonl",
    lines=True,
    nrows=2000
)

df_sample.columns

Index(['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase'],
      dtype='object')

###Convert reviews data to Parquet and select required columns



In [ ]:
reviews_in = "/content/drive/MyDrive/Kpdl1/reviews_Electronics.jsonl"
reviews_out = "/content/drive/MyDrive/Kpdl1/reviews_Electronics.parquet"

reviews = spark.read.json(reviews_in)

# Select required columns
reviews_cols = [
    "parent_asin",
    "rating",
    "text",
    "timestamp",
    "helpful_vote",
    "verified_purchase"
]
reviews = reviews.select(*reviews_cols)

# Repartition dataset to enable parallel writing
reviews = reviews.repartition(8)

# Save as Parquet
reviews.write.mode("overwrite").parquet(reviews_out)

###Load reviewdata

In [5]:
reviews = spark.read.parquet("/content/drive/MyDrive/Kpdl1/reviews_Electronics.parquet")

print("Number of rows:", reviews.count())
print("Columns:", reviews.columns)

reviews.show(5)

Number of rows: 43886944
Columns: ['parent_asin', 'rating', 'text', 'timestamp', 'helpful_vote', 'verified_purchase']
+-----------+------+--------------------+-------------+------------+-----------------+
|parent_asin|rating|                text|    timestamp|helpful_vote|verified_purchase|
+-----------+------+--------------------+-------------+------------+-----------------+
| B0C5SV6LVK|   4.0|The speakers soun...|1671477779452|           0|             true|
| B00IWANYH6|   4.0|        Worked fine.|1450891556000|           0|             true|
| B000HVHYJW|   2.0|The item arrived ...|1295754742000|           2|             true|
| B017LSIXDE|   5.0|Works perfectly a...|1531659072486|           0|             true|
| B00GFAN498|   5.0|          Works fine|1463109436000|           0|             true|
+-----------+------+--------------------+-------------+------------+-----------------+
only showing top 5 rows


In [6]:
# Check data types of each column
reviews.printSchema()

root
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



In [7]:
# Check missing values for each column
missing = reviews.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in reviews.columns
])

missing.show()

+-----------+------+----+---------+------------+-----------------+
|parent_asin|rating|text|timestamp|helpful_vote|verified_purchase|
+-----------+------+----+---------+------------+-----------------+
|          0|     0|   0|        0|           0|                0|
+-----------+------+----+---------+------------+-----------------+



In [8]:
# Check ratio of empty values in important columns
total_rows = reviews.count()

empty_parent = reviews.filter(trim(col("parent_asin")) == "").count() / total_rows
empty_text = reviews.filter(trim(col("text")) == "").count() / total_rows

{
    "empty_parent_asin_ratio": float(empty_parent),
    "empty_text_ratio": float(empty_text)
}

{'empty_parent_asin_ratio': 0.0, 'empty_text_ratio': 0.0009385934914948737}

In [9]:
# Check number of reviews before removal
before_count = reviews.count()

# Remove reviews with empty text
reviews = reviews.filter(trim(col("text")) != "")

# Check number of reviews after removal
after_count = reviews.count()

print("Reviews before:", before_count)
print("Reviews after:", after_count)
print("Removed:", before_count - after_count)

Reviews before: 43886944
Reviews after: 43845752
Removed: 41192


In [10]:
# Create review length
reviews = reviews.withColumn(
    "text_length",
    length(trim(col("text")))
)

# Statistics
reviews.select("text_length").describe().show()

+-------+------------------+
|summary|       text_length|
+-------+------------------+
|  count|          43845752|
|   mean|241.55279373472715|
| stddev| 419.5295122985488|
|    min|                 1|
|    max|             35208|
+-------+------------------+



###Remove reviews with abnormal length (<20 or >2000 characters)





In [11]:
# Remove reviews with abnormal length (<20 or >2000)

stats = reviews.select(
    count("*").alias("before"),
    sum(when((col("text_length") < 20) | (col("text_length") > 2000), 1).otherwise(0)).alias("removed")
)

result = stats.collect()[0]

before = result["before"]
removed = result["removed"]

# Filter dataset
reviews = reviews.filter(
    (col("text_length") >= 20) & (col("text_length") <= 2000)
)

after = before - removed

print("Reviews before:", before)
print("Reviews after:", after)
print("Removed:", removed)

Reviews before: 43845752
Reviews after: 38930402
Removed: 4915350


In [12]:
# Drop text_length column
reviews = reviews.drop("text_length")

In [13]:
# Rating distribution and filter valid ratings (1–5)
stats = reviews.select(
    count("*").alias("before"),
    sum(when(~col("rating").between(1,5),1).otherwise(0)).alias("removed")
).collect()[0]

before = stats["before"]
removed = stats["removed"]
after = before - removed

print("Before filter:", before)
print("After filter:", after)
print("Removed:", removed)

reviews = reviews.filter(col("rating").between(1,5))

Before filter: 38930402
After filter: 38930400
Removed: 2


In [14]:
# Remove duplicate reviews
before = reviews.count()
reviews = reviews.dropDuplicates(["parent_asin", "text"])
after = reviews.count()

print("Before:", before)
print("After:", after)
print("Removed:", before - after)

Before: 38930400
After: 38469499
Removed: 460901


In [15]:
# Normalize text
reviews = reviews.withColumn(
    "text",
    trim(
        regexp_replace(
            regexp_replace(lower(col("text")), r"[^a-z0-9\s]", ""),
            r"\s+",
            " "
        )
    )
)


In [16]:
#Convert timestap
reviews = reviews.withColumn(
    "timestamp",
    from_unixtime(col("timestamp")/1000).cast("timestamp")
)

###Save file

In [17]:
reviews = reviews.repartition(60)

# Save parquet
save_path = "/content/drive/MyDrive/Kpdl1/clean_reviews.parquet"
reviews.write.mode("overwrite").parquet(save_path)